# Qwen2.5-7B-Instruct — MLX variants summary

Loads the FP16 / 4-bit / 2-bit result JSONs from `data/`, reconstructs the `name` + `variant` fields from each run's label, and renders the KPI table + writes `summary.csv` via `bench_utils.print_results_table` and `bench_utils.dump_results_table`.

In [1]:
import json
import logging
import re
import sys
from pathlib import Path

# Make the repo root importable so we can pull the summary helpers.
REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from bench_utils import dump_results_table, print_latex_table, print_results_table

# print_results_table logs via the `bench` logger — wire it to stdout so the
# table shows up in the notebook cell output.
bench_log = logging.getLogger("bench")
bench_log.setLevel(logging.INFO)
if not bench_log.handlers:
    h = logging.StreamHandler(sys.stdout)
    h.setFormatter(logging.Formatter("%(message)s"))
    bench_log.addHandler(h)

DATA_DIR = Path.cwd() / "data"
print("data dir:", DATA_DIR)

/Users/bruski/.pyenv/versions/3.12.7/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data dir: /Users/bruski/workspace/LLM-Inference-Benchmarking-2/analysis/comparisons/data


In [2]:
# Label format from benchmark_mlx.py is `f"{variant} ({name})"`.
LABEL_RE = re.compile(r"^(?P<variant>.+?)\s*\((?P<name>[^)]+)\)\s*$")

# Preferred display order — FP16 first, then ascending bitwidth.
VARIANT_ORDER = {"FP16": 0, "MLX 4-bit": 1, "MLX 2-bit": 2}

def load_run(path: Path) -> dict:
    payload = json.loads(path.read_text())
    m = LABEL_RE.match(payload["label"])
    if not m:
        raise ValueError(f"unparseable label in {path.name}: {payload['label']!r}")
    row = dict(payload["results"])
    row["name"] = m.group("name")
    row["variant"] = m.group("variant")
    return row

rows = [load_run(p) for p in sorted(DATA_DIR.glob("*.json"))]
rows.sort(key=lambda r: (r["name"], VARIANT_ORDER.get(r["variant"], 99)))

for r in rows:
    print(f"  {r['variant']:<10}  {r['name']}")

  MLX 4-bit   Qwen2.5-7B-Instruct


In [3]:
print_results_table(rows)


  SUMMARY
Model                     Variant     GSM8K #  Accuracy %       PPL            Tok/s             TTFT ms         TPOT ms                Lat ms   Weight MB   Runtime MB     Peak MB
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Qwen2.5-7B-Instruct       MLX 4-bit        10      80.0      5.61     53.06 ± 1.16      359.20 ± 40.73    17.37 ± 0.06      4567.37 ± 996.60      4088.9        242.6      4331.5


In [4]:
# LaTeX (booktabs) rendering of the same table — paste into a paper.
# Requires \usepackage{booktabs} in the LaTeX preamble.
print_latex_table(rows, caption="Qwen2.5-7B-Instruct: MLX quantization summary.", label="tab:qwen-mlx")

\begin{table}[h]
\centering
\begin{tabular}{l l r r r r r r r r r r}
\toprule
Model & Variant & GSM8K \# & Accuracy \% & PPL & Tok/s & TTFT (ms) & TPOT (ms) & Lat (ms) & Weight MB & Runtime MB & Peak MB \\
\midrule
Qwen2.5-7B-Instruct & MLX 4-bit & 10 & 80.0 & 5.61 & $53.06 \pm 1.16$ & $359.20 \pm 40.73$ & $17.37 \pm 0.06$ & $4567.37 \pm 996.60$ & 4088.9 & 242.6 & 4331.5 \\
\bottomrule
\end{tabular}
\caption{Qwen2.5-7B-Instruct: MLX quantization summary.}
\label{tab:qwen-mlx}
\end{table}


In [5]:
# Write the union-of-keys CSV next to this notebook.
out_path = dump_results_table(str(Path.cwd()), rows, filename="qwen_summary.csv")
print("wrote:", out_path)

wrote summary csv: /Users/bruski/workspace/LLM-Inference-Benchmarking-2/analysis/comparisons/qwen_summary.csv (1 rows)
wrote: /Users/bruski/workspace/LLM-Inference-Benchmarking-2/analysis/comparisons/qwen_summary.csv


In [6]:
# Optional: preview as a pandas DataFrame for interactive filtering/sorting.
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    front = ["name", "variant", "gsm8k_accuracy", "perplexity", "throughput_tok_s", "ttft_mean_ms", "tpot_mean_ms", "weight_mem_mb", "peak_mem_mb"]
    df = df[front + [c for c in df.columns if c not in front]]
    display(df)
except ImportError:
    print("pandas not installed — skipping DataFrame preview")

,name,variant,gsm8k_accuracy,perplexity,throughput_tok_s,ttft_mean_ms,tpot_mean_ms,weight_mem_mb,peak_mem_mb,throughput_mean_tok_s,...,tpot_skipped,num_samples,total_tokens,prompt_len_mean,prompt_len_std,output_len_mean,output_len_std,gsm8k_correct,gsm8k_total,runtime_mem_mb
0,Qwen2.5-7B-Instruct,MLX 4-bit,0.8,5.609574,53.269211,359.202308,17.372519,4088.88575,4331.511406,53.058462,...,0,10,2433,95,14.506703,243.3,56.932806,8,10,242.625656
